# L1b: Choosing and Building Data Representations

Primitive values rarely appear alone. In this lab, you will organize them using tuples, arrays, sets, dictionaries, and custom composite types, choosing each representation according to the operations the problem requires.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Choose a collection:__ Distinguish tuples, arrays, sets, and dictionaries by the operations each one supports, and select among them by asking which operations the problem requires rather than which container is most familiar.
> * __Predict valid operations:__ Explain why indexing, mutation, and key lookup succeed on some containers and raise errors on others, and say which of the three a given container will accept before you run the code.
> * __Build a composite type:__ Construct immutable and mutable structs with named, typed fields, and decide which of the two a piece of data deserves based on whether it should still change after construction.

For each task: predict the result, run the cells, explain what happened, and make the requested modification.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Besides Julia's `Base` library, this lab uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks at the end. The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), which this lab does not call.

___

## Lab workflow

Keep a short record of your predictions and explanations as you work. The final check verifies that every example ran, but the important deliverable is your reasoning about why each representation supports different operations.

___


## Task 1: Tuples as fixed records
A collection type is a composite data structure aggregating multiple values, often of the same or related types, into a single container (e.g., tuples, arrays, sets, and dictionaries). It is not itself a primitive type, and its elements need not be primitives either; they may be other collections or composite types. Let's look at a few examples of collections, starting with one that we have already seen (sort of), namely [Tuples](https://docs.julialang.org/en/v1/manual/functions/#Tuples).

A tuple is an immutable, ordered collection of elements that can hold a fixed number of items, potentially of different types. Once created, a tuple's length is fixed and its slots cannot be reassigned, which makes tuples useful for grouping related values without the overhead of a mutable container. (Immutability here is shallow: if a slot holds a mutable object such as an array, that object can still be changed in place.)

> **Julia tuple memory layout:** Every tuple in Julia is an immutable composite object with a type that encodes its length and element types (e.g., `Tuple{Int64, Float64}`). The memory layout is a contiguous block of fields: if all elements are "isbits" (primitives), the tuple itself is isbits and can be unboxed (often in registers or on the stack). 
> 
> However, a non-isbits element (like a `String`) is stored as a [reference to a heap-allocated object](https://en.wikipedia.org/wiki/Pointer_(computer_programming)); any `isbits` fields beside it still sit inline. Where the tuple itself lives (register, stack, or heap) is the compiler's decision, not something the type alone determines.

Let's explore tuples with a concrete example. Since [Tuple types](https://docs.julialang.org/en/v1/base/base/#Core.Tuple) are immutable, they can't be changed once constructed.

The `example_tuple::Tuple{Int64, Float64}` variable below holds two values of different types: an age in years and a measurement. The type itself records both the length and the element types.

In [ ]:
example_tuple = let
    pair = (18,36.6); # populate with data. Notice not the same type for each element
end;

What is the type of the `example_tuple` variable? Let's use [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) to find out.

In [ ]:
typeof(example_tuple)

Tuples are immutable. Let's try to change a value in the `example_tuple::Tuple{Int64, Float64}` variable. This should blow up, because [Tuples in Julia](https://docs.julialang.org/en/v1/base/base/#Core.Tuple) are immutable.

> **Try-catch blocks:** The `try-catch` construct allows us to handle errors gracefully instead of crashing the program. 
> Code in the `try` block is executed, and if an error occurs, execution jumps to the `catch` block where we can handle the error (like printing a message) rather than terminating the program.  The program continues executing normally after the `catch` block.

So what happens?

In [ ]:
try
    example_tuple[1] = 6 # this will raise an error because tuples are immutable
catch e
    println("expected error: ", e)
end
println("After the try-catch block, the program continues executing normally.")

What does the bitstring look like for the `example_tuple::Tuple{Int64, Float64}` variable?

In [ ]:
try
    bitstring(example_tuple) # Can't get the bitstring directly; a Tuple is not a primitive type.
catch e
    println("expected error: ", e)
end

However, we can get the elements of `example_tuple` and their bit layouts by [indexing into the Tuple](https://docs.julialang.org/en/v1/base/base/#Core.Tuple). For example, let's look at the second element:

In [ ]:
bitstring(example_tuple[2]) # get the bitstring of the component i

We can see the raw bytes associated with the `example_tuple::Tuple{Int64,Float64}` using [the `reinterpret(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.reinterpret). Note: this works because the tuple is composed of `isbits` elements and the total size aligns; the exact byte order and layout you see will reflect host endianness and alignment.

In [ ]:
v = reinterpret(NTuple{16,UInt8}, example_tuple) |> collect # we have 16 8-bit blocks (128 bits total)

___

> __Checkpoint:__ Before continuing, explain why `example_tuple[1] = 6` failed while `example_tuple[2]` could still be read. Then create a second tuple containing a sensor name, a numerical reading, and a Boolean quality flag. Use [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) to inspect the resulting tuple type.

___

## Task 2: Arrays as mutable sequences
An array is a contiguous, ordered collection of elements of the same type, allowing constant-time access to its elements via integer indices. In most languages, arrays occupy a single block of memory, with element access computed as the base address plus the index times the memory size of each element.

> **Julia vs. Python arrays:** [Julia's `Array{T}` type](https://docs.julialang.org/en/v1/base/arrays/#Core.Array-Tuple%7BNothing,%20Any%7D) is a built-in, statically typed container that is `1-indexed` and stored [in column-major order](https://en.wikipedia.org/wiki/Row-_and_column-major_order). Python's native lists are heterogeneous and zero-indexed, while [NumPy's homogeneous arrays](https://numpy.org/doc/stable/reference/generated/numpy.array.html) are zero-indexed and row-major (implemented in a separate C library rather than the core language).

Arrays in both Julia and Python are mutable, meaning elements can be changed after we populate the array. Let's explore a Julia array:

In [ ]:
a = rand(10) # build a 10-element random array

We access the elements of an array by passing the index of the array in square brackets, e.g., `a[3]` returns the third element in Julia (because it is `1`-based):

In [ ]:
a[3]

Arrays are __mutable__, i.e., we can change them after we build them. For example:

In [ ]:
a[3] = π

Arrays in Julia are `1`-based, unlike C, Python, and Java, which are `0`-based.
> __Note:__ This is a deliberate choice, and Julia is in good company: Fortran, MATLAB, and R (the languages scientific computing grew up on) are all `1`-based. The practical argument is that indices line up with the mathematics you are transcribing. When you write $\sum_{i=1}^{n}a_{i}$, the loop is `for i ∈ 1:n` and `a[1]` really is $a_{1}$. The cost is real too: most algorithms in the CS literature are written `0`-based, so translating them takes care.

What happens if we try to grab an element that is _outside_ the array?

In [ ]:
try
    a[11] # asking for index 11, but the array has only 10 items
catch e
    println("expected error: ", e)
end

___

> __Checkpoint:__ Change one additional element of `a`, then retrieve it by index. Explain why the array accepted the mutation but would reject a value whose type cannot be converted to `Float64`.

___

## Task 3: Sets and dictionaries for membership and lookup
A [Set type](https://docs.julialang.org/en/v1/base/collections/#Base.Set) is an unordered collection of unique elements that supports fast membership checks, insertions, and removals. A [Dictionary (or map) is an associative container](https://docs.julialang.org/en/v1/base/collections/#Base.Dict) that stores key–value pairs, allowing lookup, insertion, and deletion of values based on their unique keys.

> **Julia vs. Python collections:** Julia's `Set{T}` and `Dict{K,V}` are parametric containers, meaning every element in a `Set` has the same type `T`, and every key–value pair in a `Dict` has types `K` and `V`. However, the elements can be any type `T`, and the keys `K` and values `V` can also be of any type. In contrast, Python's built-in `set` and `dict` are heterogeneous by default, because each slot holds a generic `object` reference. Julia can do the same with `Set{Any}` and `Dict{Any,Any}`; the difference is that Julia makes you ask for that flexibility, and hands the compiler concrete element types when you do not.

Let's build a few examples of set and dictionary collection types. The `d::Dict{Int64, String}` variable below models the lines of a text file: each key is a line number, and each value is the text on that line.

In [ ]:
d = let

    d = Dict{Int64, String}(); # creates a dictionary that models text in a file.
    d[1] = "This is the first line in a text file";
    d[2] = "This is the second line in a text file";
    d[3] = "This is the last line in a text file";

    d
end

We can access the values stored in a dictionary by passing in the `key` pointing to a `value`. Indexing with a key the dictionary does not hold raises an error, so when you are unsure, ask first with [the `haskey(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.haskey). Line `2` is there, so we can index for it directly:

In [ ]:
d[2]

Dictionaries (in general) do __not__ guarantee insertion order. Look back at the output of the cell that builds `d`: we inserted key `1` first, but it printed __last__. The iteration order comes from the hash table's internal layout, not from the order you inserted, and it can change across Julia versions or with a different set of keys, so never rely on it. If you need a map that preserves insertion order, consider `OrderedDict` from the `DataStructures.jl` package. Likewise, there is no notion of order in a set.

Consider the `s::Set{Char}` example:

In [ ]:
s = let

    s = Set{Char}(); # empty at this point
    push!(s, 'a'); # add items to the set using `push!`
    push!(s, 'b');
    push!(s, 'c');
    push!(s, 'd');

    s
end

We can't access a particular item in the `s::Set{Char}` set by passing in an index (or key) because these concepts don't apply to sets. Instead, we can use [the `pop!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pop!) to pop (get) an arbitrary element from a set:

> __Popping removes as well as returns.__ The next cell mutates `s`, so running it a second time takes a second element out. If you want to start over, re-run the cell that builds `s`.

Let's take an element out and see which one we get:

In [ ]:
pop!(s)

All the typical mathematical operations on sets, such as intersection, union, or membership checks, are implemented in most modern programming languages, including Julia; [see the documentation for operations on sets in Julia](https://docs.julialang.org/en/v1/base/collections/#Set-Like-Collections).

___

> __Checkpoint:__ Use `haskey(d, 2)` to test for a dictionary key and `'b' in s` to test set membership. Which representation would you choose for unique species names, and which would you choose for looking up a molecular weight by species name?

___

## Task 4: Custom composite types
Custom composite types are user-defined data structures that aggregate multiple fields (possibly of different types) under a single name, enabling encapsulation of related data. Think of them as custom containers that you design to hold exactly the data you need for your specific problem.

> **Language differences:** In Julia, these are declared [using the struct keyword](https://docs.julialang.org/en/v1/manual/types/#Composite-Types) with a list of named fields, similar to C. Python uses classes with attributes and methods, which is a more object-oriented approach.

We'll explore this topic in much greater depth later, but for now, let's build some simple examples to illustrate how composite types work in Julia.

In [ ]:
struct MyStudentModel

    # data -
    firstname::String # fields hold the data, they have names and types
    lastname::String
    id::Int64

    MyStudentModel(f,l,id) = new(f,l,id); # constructor
end

The `MyStudentModel(f,l,id) = new(f,l,id)` line is an [inner constructor](https://docs.julialang.org/en/v1/manual/constructors/#man-inner-constructor-methods); [`new`](https://docs.julialang.org/en/v1/manual/constructors/#man-inner-constructor-methods) is the keyword that actually builds the instance and is only available inside the type definition. Now we can create a `model::MyStudentModel` instance by calling that constructor:

In [ ]:
model = MyStudentModel("Test", "Student", 1234)

We access the data stored in our composite type using dot syntax:

In [ ]:
model.id # returns the value stored in the id field

Here's a key point: because we used the `struct` keyword, our student model is immutable. Once we build it, we cannot change any of the data stored in the model. Let's see what happens when we try:

In [ ]:
try
    model.id = 5678 # we are trying to change an immutable struct.
catch e
    println("expected error: ", e)
end

Sometimes we need to modify our data after creating it. For these cases, we can create [mutable composite types](https://docs.julialang.org/en/v1/manual/types/#Mutable-Composite-Types) by adding the `mutable` keyword when declaring the struct:

In [ ]:
mutable struct MyMutableStudentModel

     # data -
    firstname::String # fields hold the data, they have names and types
    lastname::String
    id::Int64

    MyMutableStudentModel() = new(); # builds an empty model

end

We create mutable composite types the same way as immutable ones, by calling the constructor. However, the empty constructor `new()` produces an instance whose fields are uninitialized, so you must assign every field before reading it.

The `mutable_model::MyMutableStudentModel` variable below is built that way: the `let` block constructs an empty instance, fills in each field, and returns the populated model. Wrapping this in a `let` block keeps the intermediate `model` name out of the global namespace.

In [ ]:
mutable_model = let

    model = MyMutableStudentModel(); # empty: fields are uninitialized
    model.firstname = "Firstname";
    model.lastname = "Lastname";
    model.id = 6789;

    model # return the populated model
end

___

> __Checkpoint:__ Compare `model` and `mutable_model`. Identify one engineering record that should be immutable after construction and one state object whose fields must change during a simulation.

___

## Your turn: represent a small measurement record

The default values below make the notebook runnable. Replace them with your own sample values while preserving the representation choices: a fixed student record, a mutable sequence of readings, a lookup table for metadata, and a set of quality flags. Keep the element types as they are, i.e., `Float64` readings, `Symbol` keys with `String` values in the metadata, and `Symbol` quality flags, because the check at the end tests for those types.


In [ ]:
lab_record = MyStudentModel("Ada", "Lovelace", 5800)
lab_readings = [298.15, 299.10, 300.05]
lab_metadata = Dict(:sensor => "T-101", :units => "K")
lab_quality_flags = Set([:calibrated, :finite])

(record = lab_record, readings = lab_readings, metadata = lab_metadata, flags = lab_quality_flags)

___

## Lab check

Run this cell after completing the notebook. These checks verify the representations and mutations used above; your checkpoint explanations provide the reasoning the tests cannot capture.


In [ ]:
let
    @testset verbose = true "CHEME 4/5800 L1b Representation Check" begin
        @test example_tuple isa Tuple{Int64,Float64}
        @test a isa Vector{Float64}
        @test a[3] == Float64(π)
        @test d isa Dict{Int64,String}
        @test d[2] == "This is the second line in a text file"
        @test s isa Set{Char}
        @test length(s) == 3
        @test model isa MyStudentModel
        @test model.id == 1234
        @test mutable_model isa MyMutableStudentModel
        @test mutable_model.id == 6789
        @test lab_record isa MyStudentModel
        @test lab_readings isa Vector{Float64}
        @test lab_metadata isa Dict{Symbol,String}
        @test lab_quality_flags isa Set{Symbol}
    end
end;

___


## Summary

Choosing a representation means choosing which operations the program should support.

> __Key Takeaways:__
>
> * **A container is a set of operations:** Tuples fix their contents at construction, arrays give ordered indexed access you can overwrite, sets answer membership without order, and dictionaries answer lookup by key, so naming the operation you need also names the container you need.
> * **Mutability is a design decision the compiler can enforce:** Julia separates the immutable and mutable cases into different types rather than different conventions, so a field that must not be reassigned after construction is guaranteed by the type rather than by discipline.
> * **A composite type gives your data a vocabulary:** A `struct` replaces positional bookkeeping with named, typed fields and a type of its own, which is what lets later code dispatch on the thing instead of picking apart its pieces.

Week 2 builds on these representations by defining interfaces and performing transformations, grouping, and summarization.
___